In [12]:
import zipfile, json

z = zipfile.ZipFile("../data/raw/osv/pypi.zip")
names = z.namelist()

print(len(names), "files")
print(names[:5])

adv = json.loads(z.read(names[0]))
print(json.dumps(adv, indent=2))

25726 files
['GHSA-226f-f24g-524w.json', 'GHSA-227r-w5j2-6243.json', 'GHSA-22c2-9gwg-mj59.json', 'GHSA-22cc-w7xm-rfhx.json', 'GHSA-22cj-m4wf-fv2c.json']
{
  "schema_version": "1.9.0",
  "id": "GHSA-226f-f24g-524w",
  "published": "2026-06-17T14:10:56Z",
  "modified": "2026-07-13T16:42:55.257503767Z",
  "aliases": [
    "CVE-2026-54008",
    "PYSEC-2026-2690"
  ],
  "summary": "Open WebUI: Redirect-Bypass SSRF in OAuth `_process_picture_url` (incomplete-fix sibling of CVE-2026-45401)",
  "details": "## Summary\n\n`backend/open_webui/utils/oauth.py::_process_picture_url` (v0.9.5, lines 1435-1470) calls `validate_url(picture_url)` on the initial URL only, then invokes `aiohttp.ClientSession.get(picture_url, ...)` without `allow_redirects=False`. aiohttp's default is `allow_redirects=True, max_redirects=10`; the function does not pass the project's `AIOHTTP_CLIENT_ALLOW_REDIRECTS` env constant either. An attacker with a valid OAuth IdP identity can therefore submit a public URL that 302-re

In [13]:
import zipfile
import pandas as pd

z_npm = zipfile.ZipFile("../data/raw/osv/npm.zip")
names_npm = z_npm.namelist()

npm_names = pd.Series(names_npm)
print(npm_names.head())

0     EEF-CVE-2026-56812.json
1    GHSA-2234-fmw7-43wr.json
2    GHSA-2238-xc5r-v9hj.json
3    GHSA-223g-f5mq-gw33.json
4    GHSA-223j-4rm8-mrmf.json
dtype: object


In [14]:
import zipfile
import pandas as pd

z_npm = zipfile.ZipFile("../data/raw/osv/npm.zip")
names_npm = z_npm.namelist()

npm_names = pd.Series(names_npm)
v = npm_names.str.split("-").str[0]
print(v.value_counts())

MAL     221943
GHSA      7465
EEF          1
GSD          1
Name: count, dtype: int64


In [15]:
import zipfile
import pandas as pd

z_npm = zipfile.ZipFile("../data/raw/osv/pypi.zip")
names_pypi = z_npm.namelist()

pypi_names = pd.Series(names_pypi)
v = pypi_names.str.split("-").str[0]
print(v.value_counts())

MAL      11774
PYSEC     7608
GHSA      6336
OSV          8
Name: count, dtype: int64


In [16]:
import zipfile
import json
import pandas as pd


def parse_advisory(adv, ecosystem):
    """Turn one OSV advisory into rows: one row per affected package."""
    rows = []

    if adv["id"].startswith("MAL-"):
        return []

    if adv.get("withdrawn"):
        return []

    for entry in adv.get("affected", []):
        # Some entries only name a git repo, not a package (e.g. EEF-CVE-2026-56812,
        # the Phoenix framework). Without a package name we can't label anything.
        if "package" not in entry:
            continue
        entry_eco = entry["package"]["ecosystem"]
        name_pk = entry["package"]["name"]
        if entry_eco != ecosystem:
            continue

        fixed_versions = []
        for rng in entry.get("ranges", []):
            if rng["type"] == "GIT":
                continue
            for event in rng.get("events", []):
                if "fixed" in event:
                    fixed_versions.append(event["fixed"])

        row = {
            "id": adv["id"],
            "ecosystem": entry_eco,
            "package": name_pk,
            "published": adv["published"],
            "fixed_versions": fixed_versions,
            "aliases": adv.get("aliases", []),
        }
        rows.append(row)

    return rows
    


In [17]:
#These are small hand-made advisories, one for each tricky case.

t_mal = {"id": "MAL-2024-1", "published": "2024-01-01T00:00:00Z",
         "affected": [{"package": {"ecosystem": "npm", "name": "reqeusts"}}]}

t_withdrawn = {"id": "GHSA-aaaa", "published": "2024-01-01T00:00:00Z",
               "withdrawn": "2024-02-01T00:00:00Z",
               "affected": [{"package": {"ecosystem": "npm", "name": "lodash"},
                             "ranges": [{"type": "SEMVER",
                                         "events": [{"introduced": "0"}, {"fixed": "4.17.21"}]}]}]}

t_two = {"id": "GHSA-bbbb", "published": "2024-03-01T00:00:00Z", "aliases": ["CVE-2024-1"],
         "affected": [
             {"package": {"ecosystem": "npm", "name": "pkg-a"},
              "ranges": [{"type": "SEMVER", "events": [{"introduced": "0"}, {"fixed": "1.2.3"}]}]},
             {"package": {"ecosystem": "npm", "name": "pkg-b"},
              "ranges": [{"type": "GIT", "events": [{"introduced": "0"}, {"fixed": "abc123def"}]},
                         {"type": "SEMVER", "events": [{"introduced": "0"}, {"fixed": "2.0.0"}]}]},
         ]}

t_nopkg = {"id": "EEF-test", "published": "2024-01-01T00:00:00Z",
           "affected": [{"ranges": [{"type": "GIT",
                                     "events": [{"introduced": "0"}, {"fixed": "abc"}]}]}]}

print(parse_advisory(t_mal, "npm") == [])
print(parse_advisory(t_nopkg, "npm") == [])       # entry without a package, skipped
print(parse_advisory(t_withdrawn, "npm") == [])
print(parse_advisory(t_two, "PyPI") == [])        # wrong ecosystem, so nothing

rows = parse_advisory(t_two, "npm")
print(len(rows) == 2)
print(rows[0]["package"] == "pkg-a" and rows[0]["fixed_versions"] == ["1.2.3"])
print(rows[1]["fixed_versions"] == ["2.0.0"])     # the GIT hash was skipped
print(rows[1]["aliases"] == ["CVE-2024-1"])

True
True
True
True
True
True
True
True


In [18]:
def load_osv_zip(path, ecosystem):
    """Read every advisory in an OSV zip file and return one DataFrame."""
    z = zipfile.ZipFile(path)
    all_rows = []

    for name in z.namelist():
        adv = json.loads(z.read(name))
        all_rows.extend(parse_advisory(adv, ecosystem))
        

    df = pd.DataFrame(all_rows)

    df["published"] = pd.to_datetime(df["published"], utc=True, format="ISO8601")


    return df


In [19]:
pypi = load_osv_zip("../data/raw/osv/pypi.zip", "PyPI")
npm = load_osv_zip("../data/raw/osv/npm.zip", "npm")


print(len(pypi), "PyPI rows")
print(len(npm), "npm rows")
pypi.head()

18448 PyPI rows
9675 npm rows


,id,ecosystem,package,published,fixed_versions,aliases
0,GHSA-226f-f24g-524w,PyPI,open-webui,2026-06-17 14:10:56+00:00,[0.9.6],"[CVE-2026-54008, PYSEC-2026-2690]"
1,GHSA-227r-w5j2-6243,PyPI,invokeai,2025-03-20 12:32:41+00:00,[5.3.0rc1],"[CVE-2024-11042, PYSEC-2026-358]"
2,GHSA-22c2-9gwg-mj59,PyPI,langroid,2025-05-20 18:01:52+00:00,[0.53.15],"[CVE-2025-46725, PYSEC-2026-1531]"
3,GHSA-22cc-w7xm-rfhx,PyPI,mezzanine,2024-02-28 21:30:20+00:00,[],"[CVE-2024-25170, PYSEC-2026-1626]"
4,GHSA-22cj-m4wf-fv2c,PyPI,praisonai,2026-06-18 13:52:32+00:00,[4.6.59],"[CVE-2026-56833, PYSEC-2026-3499]"


In [20]:
pypi[pypi["id"] == "PYSEC-2026-2690"]

,id,ecosystem,package,published,fixed_versions,aliases
16340,PYSEC-2026-2690,PyPI,open-webui,2026-07-13 15:46:19.081114+00:00,[0.9.6],"[CVE-2026-54008, GHSA-226f-f24g-524w]"


In [21]:
pypi[pypi["id"].isin(["GHSA-226f-f24g-524w", "PYSEC-2026-2690"])]

,id,ecosystem,package,published,fixed_versions,aliases
0,GHSA-226f-f24g-524w,PyPI,open-webui,2026-06-17 14:10:56+00:00,[0.9.6],"[CVE-2026-54008, PYSEC-2026-2690]"
16340,PYSEC-2026-2690,PyPI,open-webui,2026-07-13 15:46:19.081114+00:00,[0.9.6],"[CVE-2026-54008, GHSA-226f-f24g-524w]"


In [24]:
pypi = load_osv_zip("../data/raw/osv/pypi.zip", "PyPI")

def combine_lists(lists):
    """Turn several lists into one list."""
    result = []
    for lst in lists : 
        result.extend(lst)

    return result


pypi_merged = pypi.groupby(["id", "package"]).agg(
    published=("published", "first"),
    fixed_versions=("fixed_versions", combine_lists),
    ecosystem =("ecosystem", "first"),
    aliases=("aliases", "first") 
).reset_index()

print(len(pypi_merged))

14467


In [25]:
pypi_merged[pypi_merged["id"] == "GHSA-r8qr-wwg3-2r85"]

,id,package,published,fixed_versions,ecosystem,aliases
5919,GHSA-r8qr-wwg3-2r85,saleor,2023-03-03 22:46:04+00:00,"[3.1.48, 3.11.12, 3.10.14, 3.9.27, 3.8.30, 3.7...",PyPI,"[CVE-2023-26051, PYSEC-2026-917]"


In [26]:

def combine_lists(lists):
    """Turn several lists into one list."""
    result = []
    for lst in lists : 
        result.extend(lst)

    return result


npm_merged = npm.groupby(["id", "package"]).agg(
    published=("published", "first"),
    fixed_versions=("fixed_versions", combine_lists),
    ecosystem =("ecosystem", "first"),
    aliases=("aliases", "first") 
).reset_index()

print(len(npm_merged))

7719


In [35]:
def bug_key(row):
    """Return one shared name for a bug, so copies from different databases match."""
    if row["id"].startswith("GHSA-"):
        return row["id"]
    for alias in row["aliases"]:
        if alias.startswith("GHSA-"):
            return alias
    return row["id"]

pypi_merged["bug_key"] = pypi_merged.apply(bug_key, axis=1)

pypi_bugs = pypi_merged.groupby(["package", "bug_key"]).agg(
    published=("published", "min"),
    fixed_versions=("fixed_versions", combine_lists),
    ecosystem=("ecosystem", "first"),
    ids=("id", list),
).reset_index()

print(len(pypi_bugs))

7778


In [36]:
pypi_bugs[pypi_bugs["package"] == "trac"]

,package,bug_key,published,fixed_versions,ecosystem,ids
7157,trac,GHSA-2q26-r8c4-jfx5,2006-11-14 19:07:00+00:00,"[0.10.1, 0.11]",PyPI,"[GHSA-2q26-r8c4-jfx5, PYSEC-2006-3]"
7158,trac,GHSA-437p-qw95-wqqr,2008-12-17 18:30:00+00:00,"[0.11.2, 0.11.2]",PyPI,"[GHSA-437p-qw95-wqqr, PYSEC-2008-6]"
7159,trac,GHSA-6vhp-hp77-6w52,2005-12-31 05:00:00+00:00,"[0.9-stable, 0.10]",PyPI,"[GHSA-6vhp-hp77-6w52, PYSEC-2005-1]"
7160,trac,GHSA-7jjr-3r8r-9pcf,2007-03-10 22:19:00+00:00,"[0.10.3.1, 0.10.3.1]",PyPI,"[GHSA-7jjr-3r8r-9pcf, PYSEC-2007-3]"
7161,trac,GHSA-f9qv-j5g6-g5cr,2009-12-23 21:30:00+00:00,"[0.11.6, 0.11.6]",PyPI,"[GHSA-f9qv-j5g6-g5cr, PYSEC-2009-7]"
7162,trac,GHSA-r524-c2gf-5chr,2006-07-21 14:03:00+00:00,"[0.9.6, 0.9.6]",PyPI,"[GHSA-r524-c2gf-5chr, PYSEC-2006-2]"
7163,trac,GHSA-rcmj-xp8f-f6q4,2008-07-27 22:41:00+00:00,"[0.10.5, 0.10.5]",PyPI,"[GHSA-rcmj-xp8f-f6q4, PYSEC-2008-4]"
7164,trac,GHSA-w7x2-57f7-3p3x,2007-03-10 22:19:00+00:00,"[0.10.3.1, 0.10.3.1]",PyPI,"[GHSA-w7x2-57f7-3p3x, PYSEC-2007-2]"
7165,trac,GHSA-ww53-wxxr-8f9w,2008-12-17 18:30:00+00:00,"[0.11.2, 0.11.2]",PyPI,"[GHSA-ww53-wxxr-8f9w, PYSEC-2008-7]"
7166,trac,GHSA-x6jf-c7wh-7m7w,2008-07-27 22:41:00+00:00,"[0.10.5, 0.10.5]",PyPI,"[GHSA-x6jf-c7wh-7m7w, PYSEC-2008-5]"


In [38]:
npm_merged["bug_key"] = npm_merged.apply(bug_key, axis=1)

npm_bugs = npm_merged.groupby(["package", "bug_key"]).agg(
    published=("published", "min"),
    fixed_versions=("fixed_versions", combine_lists),
    ecosystem=("ecosystem", "first"),
    ids=("id", list),
).reset_index()

print(len(npm_bugs))

7718


In [39]:
print(pypi_bugs.columns)

Index(['package', 'bug_key', 'published', 'fixed_versions', 'ecosystem',
       'ids'],
      dtype='object')
